# 温度传感器测试及半导体致冷控温实验 (最小二乘法线性拟合)

本 Notebook 用于处理 **FD-TM 温度传感器测试及半导体致冷控温实验** 数据。实验通过 TCF-708 智能温度调节仪及半导体致冷堆（帕尔贴效应）在低于环境温度（致冷段）和高于环境温度（加热段）下精确控温，测量集成电路温度传感器 **AD590** 在不同温度 $t$ 下的取样电阻电压 $U_0$ 与输出电流 $I_0$，利用**最小二乘法线性回归**求解传感器的电流温度灵敏度（斜率 $B$）、特征温度截距、线性相关系数 $r$，并完成斜率标准不确定度评定与规范修约。

> **仪器与测量说明**：
> * **TCF-708 智能控温仪**：分辨率 $0.1\,^\circ\mathrm{C}$，经 P.I.D 自适应整定与“UU”值微调后控温精度达 $\pm 0.1\,^\circ\mathrm{C}$。
> * **4 1/2 数字电压表**：量程 $0 \sim 1.9999\,\mathrm{V}$，分辨率 $0.0001\,\mathrm{V}$，测量精度 $\pm 0.1\%$。
> * **标准取样电阻**：$R = 1.00\,\mathrm{k\Omega}$（$\pm 0.1\,\Omega / 25\,^\circ\mathrm{C}$），输出电流换算关系为 $I_0\,(\mu\mathrm{A}) = \frac{U_0\,(\mathrm{V})}{1.00\,\mathrm{k\Omega}} \times 1000 = 1000 U_0$。

---
### 实验原理与数学公式

#### 1. AD590 温度传感器测温原理与特性方程
AD590 是电流型集成电路温度传感器，在 $+4 \sim +30\,\mathrm{V}$ 工作电压范围内等效于高阻抗恒流源。其输出电流 $I_0\,(\mu\mathrm{A})$ 与热力学温度 $T\,(\mathrm{K})$ 严格成正比，理论电流温度系数（灵敏度）为：
$$K_I = \frac{I_0}{T} = \frac{3k}{qR} \ln 8 = 1.000\,\mu\mathrm{A/K}$$
转换为摄氏温标 $t\,(^\circ\mathrm{C}) = T - 273.15$，输出电流与摄氏温度满足线性关系：
$$I_0 = B \cdot t + A$$
* **斜率 $B$**：传感器的电流温度系数（灵敏度），理论值 $B_0 = 1.000\,\mu\mathrm{A/K}$（对应 $1.00\,\mathrm{k\Omega}$ 取样电阻上的电压灵敏度 $k_U = 1.000\,\mathrm{mV/K} = 0.001000\,\mathrm{V/^\circ C}$）。
* **截距 $A$**：$0.00\,^\circ\mathrm{C}$（$273.15\,\mathrm{K}$）时的输出电流，理论值 $A_0 = 273.15\,\mu\mathrm{A}$（对应电压 $0.27315\,\mathrm{V}$）。
* **$25.00\,^\circ\mathrm{C}$ 定标值**：AD590 在 $298.15\,\mathrm{K}$（$+25.00\,^\circ\mathrm{C}$）标准下激光校准，满足 $I_0 = B(t - 25.00) + I_{25\,^\circ\mathrm{C}}$，理论定标电流 $I_{25\,^\circ\mathrm{C}} = 298.15\,\mu\mathrm{A}$（对应电压 $0.29815\,\mathrm{V}$）。

#### 2. 最小二乘法一元线性回归与不确定度评定
设共测量 $n$ 组温度与输出电流数据 $(t_i, I_i)$，定义均值 $\bar{t}, \bar{I}$ 及离差平方和：
$$L_{tt} = \sum_{i=1}^n (t_i - \bar{t})^2, \quad L_{II} = \sum_{i=1}^n (I_i - \bar{I})^2, \quad L_{tI} = \sum_{i=1}^n (t_i - \bar{t})(I_i - \bar{I})$$
* **拟合斜率（灵敏度 $B$）与截距 $A$**：
  $$B = \frac{L_{tI}}{L_{tt}}, \quad A = \bar{I} - B \cdot \bar{t}$$
* **线性相关系数 $r$**：
  $$r = \frac{L_{tI}}{\sqrt{L_{tt} \cdot L_{II}}}$$
* **斜率标准不确定度 $u(B)$ 与相对不确定度 $u_r(B)$**：
  $$u(B) = |B| \sqrt{\frac{1/r^2 - 1}{n - 2}}, \quad u_r(B) = \frac{u(B)}{|B|} \times 100\%$$
* **科学修约**：调用 `scientific_round` 严格执行国家计量规范（JJF 1059）“四舍六入五凑偶”修约规则，不确定度保留 1 位有效数字，灵敏度主值末位与不确定度末位严格对齐。

#### 3. 半导体致冷堆原理与摄氏温标非平衡电桥调零
* **帕尔贴（Peltier）效应**：N 型和 P 型半导体电偶对通以直流电（N 极流向 P 极）时，载流子流过结点势能改变，冷端吸热、热端放热；电流反向则冷热端互换。
* **非平衡电桥技术调零**：由于 AD590 在 $0\,^\circ\mathrm{C}$ 时存在 $273.15\,\mu\mathrm{A}$ 的基准电流，实际摄氏温度计应用中将其接入非平衡电桥，在 $0\,^\circ\mathrm{C}$ 下调节可变电阻 $R_x$ 使电桥输出电压为 $0\,\mathrm{V}$，从而实现摄氏温度直读。

In [1]:
from decimal import Decimal

from python.utils import scientific_round, linear_regression

print("温度传感器测试及半导体致冷控温实验模块加载完成。")

温度传感器测试及半导体致冷控温实验模块加载完成。


### 1. 实验参数与原始测量数据输入
> **数据配置说明**：
> * 下方已预置实验报告记录的**半导体致冷段**（$15\,^\circ\mathrm{C} \to -1.8\,^\circ\mathrm{C}$，共 7 组）与**加热段**（$20\,^\circ\mathrm{C} \to 80\,^\circ\mathrm{C}$，共 7 组）共 14 组电压测量值 `U_0`（单位：V）。
> * 保持 `Decimal("...")` 字符串初始化以确保全链路高精度运算。

In [2]:
# --- 实验电路与传感器理论定标参数 ---
R_LOAD_KOHM = Decimal("1.00")            # 取样标准电阻 R = 1.00 kΩ (即 1000 Ω)
T_KELVIN_OFFSET = Decimal("273.15")      # 摄氏温度转热力学温度常数 (K)
B_THEORY = Decimal("1.000")              # AD590 理论电流温度系数 (灵敏度) K_I = 1.000 μA/K
I_0C_THEORY = Decimal("273.15")          # 0.00 °C (273.15 K) 理论输出电流 (μA)
I_25C_THEORY = Decimal("298.15")         # 25.00 °C (298.15 K) 定标点理论输出电流 (μA)

# --- 原始测量数据：(温度 t / °C, 电压 U_0 / V) ---
# 1. 半导体致冷温区测量数据 (15 °C 降至 -1.8 °C，共 7 组)
cooling_raw = [
    (Decimal("15"),   Decimal("0.2872")),
    (Decimal("12"),   Decimal("0.2838")),
    (Decimal("9"),    Decimal("0.2803")),
    (Decimal("6"),    Decimal("0.2773")),
    (Decimal("3"),    Decimal("0.2737")),
    (Decimal("0"),    Decimal("0.2705")),
    (Decimal("-1.8"), Decimal("0.2659")),
]

# 2. 加热温区测量数据 (20 °C 升至 80 °C，共 7 组)
heating_raw = [
    (Decimal("20"), Decimal("0.2942")),
    (Decimal("30"), Decimal("0.3041")),
    (Decimal("40"), Decimal("0.3142")),
    (Decimal("50"), Decimal("0.3242")),
    (Decimal("60"), Decimal("0.3351")),
    (Decimal("70"), Decimal("0.3440")),
    (Decimal("80"), Decimal("0.3538")),
]

# 按温度由低到高整合全温区数据 (-1.8 °C ~ 80 °C，共 14 组)
all_raw = sorted(cooling_raw + heating_raw, key=lambda item: item[0])

print(f"取样电阻 R          : {R_LOAD_KOHM} kΩ")
print(f"理论电流温度系数 B_0: {B_THEORY} μA/K (对应电压灵敏度 1.000 mV/K)")
print(f"已加载致冷温区数据  : {len(cooling_raw)} 组 (-1.8 °C ~ 15 °C)")
print(f"已加载加热温区数据  : {len(heating_raw)} 组 (20 °C ~ 80 °C)")
print(f"全温区合计数据点数  : {len(all_raw)} 组")

取样电阻 R          : 1.00 kΩ
理论电流温度系数 B_0: 1.000 μA/K (对应电压灵敏度 1.000 mV/K)
已加载致冷温区数据  : 7 组 (-1.8 °C ~ 15 °C)
已加载加热温区数据  : 7 组 (20 °C ~ 80 °C)
全温区合计数据点数  : 14 组


### 2. 数据转换与温度-电压-电流对照表

In [3]:
# --- 数据预处理与物理量转换 ---
# 输出电流 I_0 (μA) = U_0 (V) / R (kΩ) * 1000
print("=" * 74)
print("           AD590 温度传感器输出特性原始数据与电流换算表           ")
print("=" * 74)
header = f"{'序号':^4}|{'温区类型':^8}|{'摄氏温度 t (°C)':^14}|{'热力学温度 T (K)':^15}|{'输出电压 U_0 (V)':^15}|{'输出电流 I_0 (μA)':^15}"
print(header)
print("-" * 74)

cooling_set = set(cooling_raw)
for idx, (t_c, u_v) in enumerate(all_raw, start=1):
    region = "致冷段" if (t_c, u_v) in cooling_set else "加热段"
    t_k = t_c + T_KELVIN_OFFSET
    i_ua = (u_v / R_LOAD_KOHM) * Decimal("1000")
    print(f"{idx:^6}|{region:^11}|{str(t_c):^16}|{t_k:^17.2f}|{u_v:^17.4f}|{i_ua:^17.1f}")

print("=" * 74)

           AD590 温度传感器输出特性原始数据与电流换算表           
 序号 |  温区类型  | 摄氏温度 t (°C)  |  热力学温度 T (K)  | 输出电压 U_0 (V)  | 输出电流 I_0 (μA) 
--------------------------------------------------------------------------
  1   |    致冷段    |      -1.8      |     271.35      |     0.2659      |      265.9      
  2   |    致冷段    |       0        |     273.15      |     0.2705      |      270.5      
  3   |    致冷段    |       3        |     276.15      |     0.2737      |      273.7      
  4   |    致冷段    |       6        |     279.15      |     0.2773      |      277.3      
  5   |    致冷段    |       9        |     282.15      |     0.2803      |      280.3      
  6   |    致冷段    |       12       |     285.15      |     0.2838      |      283.8      
  7   |    致冷段    |       15       |     288.15      |     0.2872      |      287.2      
  8   |    加热段    |       20       |     293.15      |     0.2942      |      294.2      
  9   |    加热段    |       30       |     303.15      |     0.3041      |      30

### 3. 最小二乘法线性拟合、灵敏度不确定度评定与规范修约输出
> **分析说明**：
> 讲义要求采用最小二乘法对 AD590 的电流-温度曲线进行直线拟合，求出灵敏度（斜率 $B$）、截距 $A$ 及相关系数 $r$。下方代码同时输出**全温区（$-1.8\,^\circ\mathrm{C} \sim 80\,^\circ\mathrm{C}$）**、**加热温区（$20\,^\circ\mathrm{C} \sim 80\,^\circ\mathrm{C}$）**与**致冷温区（$-1.8\,^\circ\mathrm{C} \sim 15\,^\circ\mathrm{C}$）**的完整回归结果及修约表达式。

In [4]:
# --- 最小二乘法线性回归分析：I_0 (μA) = B * t (°C) + A ---
def analyze_temperature_segment(segment_name, data_pairs):
    # 按温度升序排列
    sorted_pairs = sorted(data_pairs, key=lambda x: x[0])
    t_list = [p[0] for p in sorted_pairs]
    u_list = [p[1] for p in sorted_pairs]
    i_list = [(u / R_LOAD_KOHM) * Decimal("1000") for u in u_list]

    # 调用线性回归工具函数: I_0 = B * t + A
    B_slope, A_intercept, r_corr, u_B = linear_regression(t_list, i_list)

    # 相对不确定度与科学修约
    u_r_B = (u_B / abs(B_slope)) * Decimal("100")
    B_final, u_B_final = scientific_round(B_slope, u_B)

    # 换算为电压对温度的斜率 k_U (V/°C)
    k_u_v = B_slope / Decimal("1000")
    u_k_u_v = u_B / Decimal("1000")
    k_u_final, u_ku_final = scientific_round(k_u_v, u_k_u_v)

    # 25.00 °C 定标点拟合电流值 I(25°C) = 25 * B + A
    i_25c_fit = B_slope * Decimal("25.00") + A_intercept

    # 与理论值的相对误差
    err_B = (abs(B_slope - B_THEORY) / B_THEORY) * Decimal("100")
    err_A0 = (abs(A_intercept - I_0C_THEORY) / I_0C_THEORY) * Decimal("100")
    err_A25 = (abs(i_25c_fit - I_25C_THEORY) / I_25C_THEORY) * Decimal("100")

    print("=" * 64)
    print(f"  【{segment_name}】 最小二乘法线性拟合与不确定度评定结果 (n = {len(t_list)})")
    print("=" * 64)
    print(f"拟合直线方程 (电流型)   : I_0 = {B_slope:.4f} * t + {A_intercept:.2f} (μA)")
    print(f"拟合直线方程 (电压型)   : U_0 = {k_u_v:.6f} * t + {A_intercept/Decimal('1000'):.5f} (V)")
    print("-" * 64)
    print(f"电流温度灵敏度 B (斜率) : {B_slope:.4f} μA/K (理论值 {B_THEORY} μA/K)")
    print(f"斜率标准不确定度 u(B)   : {u_B:.4f} μA/K")
    print(f"斜率相对不确定度 u_r(B) : {u_r_B:.2f}%")
    print(f"灵敏度相对误差 E_r(B)   : {err_B:.2f}%")
    print("-" * 64)
    print(f"0 °C 拟合截距 A (0°C)   : {A_intercept:.2f} μA (理论值 {I_0C_THEORY} μA, 相对误差 {err_A0:.2f}%)")
    print(f"25 °C 定标拟合值 I(25°C): {i_25c_fit:.2f} μA (理论值 {I_25C_THEORY} μA, 相对误差 {err_A25:.2f}%)")
    print(f"线性相关系数 r          : {r_corr:.6f}")
    print("-" * 64)
    print(f"★ 电流灵敏度规范修约    : B = ({B_final} ± {u_B_final}) μA/K")
    print(f"★ 电压灵敏度规范修约    : k_U = ({k_u_final} ± {u_ku_final}) V/°C")
    print("=" * 64 + "\n")

# 依次输出：1. 全温区合并拟合；2. 加热段独立拟合；3. 致冷段独立拟合
analyze_temperature_segment("全温区 (-1.8 °C ~ 80 °C)", all_raw)
analyze_temperature_segment("加热温区 (20 °C ~ 80 °C)", heating_raw)
analyze_temperature_segment("致冷温区 (-1.8 °C ~ 15 °C)", cooling_raw)

  【全温区 (-1.8 °C ~ 80 °C)】 最小二乘法线性拟合与不确定度评定结果 (n = 14)
拟合直线方程 (电流型)   : I_0 = 1.0584 * t + 270.87 (μA)
拟合直线方程 (电压型)   : U_0 = 0.001058 * t + 0.27087 (V)
----------------------------------------------------------------
电流温度灵敏度 B (斜率) : 1.0584 μA/K (理论值 1.000 μA/K)
斜率标准不确定度 u(B)   : 0.0138 μA/K
斜率相对不确定度 u_r(B) : 1.31%
灵敏度相对误差 E_r(B)   : 5.84%
----------------------------------------------------------------
0 °C 拟合截距 A (0°C)   : 270.87 μA (理论值 273.15 μA, 相对误差 0.84%)
25 °C 定标拟合值 I(25°C): 297.33 μA (理论值 298.15 μA, 相对误差 0.28%)
线性相关系数 r          : 0.998976
----------------------------------------------------------------
★ 电流灵敏度规范修约    : B = (1.06 ± 0.01) μA/K
★ 电压灵敏度规范修约    : k_U = (0.00106 ± 0.00001) V/°C

  【加热温区 (20 °C ~ 80 °C)】 最小二乘法线性拟合与不确定度评定结果 (n = 7)
拟合直线方程 (电流型)   : I_0 = 0.9982 * t + 274.32 (μA)
拟合直线方程 (电压型)   : U_0 = 0.000998 * t + 0.27432 (V)
----------------------------------------------------------------
电流温度灵敏度 B (斜率) : 0.9982 μA/K (理论值 1.000 μA/K)
斜率标准不确定度 u(B)   : 0.0085 μA/K
